# COMP9085 Week 7 Laboratory
## Protein representations, dimensionality reduction and k-NN: classical features versus ESM-2

**Two hours.** Last session you built k-NN and found that everything depends on the space in
which "nearest" is measured. Today you build three kinds of space for the same 2,000 real human
proteins (hand-made descriptors, a pretrained protein language-model embedding, and reduced
versions of both) and find out, with a controlled experiment, which one gives k-NN the more
useful neighbours, for which biological question, and why.

**The data.** 2,000 reviewed human proteins from **UniProtKB/Swiss-Prot**
(release 2026_03), one per UniRef50 cluster, 50–1,000 residues. Two
labels from UniProt keywords: **signal peptide** (KW-0732; 350 proteins) and
**transmembrane** (KW-0812; 488). Each protein also has an **ESM-2 embedding**
(`facebook/esm2_t6_8M_UR50D`, 8 million parameters, 320 numbers), computed once by the module
team from the real sequence and from a residue-shuffled copy. Everything is in `data/`:
`data/README.md` documents every column and `data/PROVENANCE.json` the source, the model
version and checksums. **No network, GPU or deep-learning package is needed.**

**The question.** If k-NN predicts a protein's annotation from its nearest neighbours, does it
find better neighbours among proteins with similar *composition and hydropathy*, or among
proteins with similar *ESM-2 embeddings*? And what happens to the neighbours when we reduce
dimension with PCA, t-SNE or UMAP?

**By the end of this lab you should be able to**
1. explore a real protein dataset and say where its labels come from and how reliable they are;
2. compute classical sequence descriptors and say what each can and cannot see;
3. apply PCA correctly and interpret the spectrum, the loadings and the effect of scaling;
4. run and read t-SNE and UMAP, and measure how faithful a 2-D map is;
5. use pretrained ESM-2 embeddings in a downstream k-NN task, with preprocessing in the folds;
6. compare two representations fairly, select hyperparameters by cross-validation, evaluate once,
   and interpret the difference with its uncertainty and its biological limits.

**Time plan.** Tasks 1–2 about 20 min · Tasks 3–5 about 35 min · Task 6 about 30 min ·
Tasks 7–9 about 35 min. **Tasks 1–9 are the core lab.** Four optional extensions follow.

**How the cells work.** Cells marked *provided* run as they are: read them, then run them.
Cells containing `# your code here` are yours; nothing later depends on them, so the notebook
still runs if you skip one. Some tasks ask you to **predict** first: write the prediction
down before running anything. Markdown cells marked *Your answer here* are for written
answers, using your own numbers.

**Rules for the whole lab.** The proteins are split once, in a provided cell, into 1,500
training and 500 test proteins (the Week 7 split). Every choice is made with 5-fold
cross-validation on the training proteins. The test set is used **once**, in Task 6. Anything
that learns from data (scaler, PCA) sits inside a `Pipeline`, so it is refitted in every fold.

**The Week 7 page is your interactive companion**
(https://hafli.github.io/comp9085-ml-biology/weeks/week07.html). Its section 11 maps each task
to the exercise that shows the same idea with the same numbers.

**Practice, not assessment.** This lab is not part of Assignment 1 and uses different data.

**Deliverable.** This notebook, run top to bottom (**Kernel → Restart & Run All**), with every
question answered in the markdown cells.

### Setup (provided)

Run this cell first. `umap-learn` is optional: if it is not installed, the UMAP cells are skipped (`pip install umap-learn` to add it).

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness
from sklearn.metrics import (roc_auc_score, average_precision_score, confusion_matrix,
                             ConfusionMatrixDisplay, precision_score, recall_score, f1_score)
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import sklearn
try:
    import umap
    HAVE_UMAP = True
except ImportError:
    HAVE_UMAP = False
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__,
      "| umap-learn", umap.__version__ if HAVE_UMAP else "not installed")

warnings.filterwarnings("ignore")
SEED = 42
DATA = Path("data")
plt.rcParams["figure.dpi"] = 90

## Task 1. Explore the protein dataset (about 15 minutes)

Start with the data, not the model. Load the proteins and the two embedding files (provided).

In [ ]:
# provided
prot = pd.read_csv(DATA / "swissprot_human_2000.csv")
emb = pd.read_csv(DATA / "esm2_t6_8M_embeddings.csv.gz").set_index("accession")
emb_shuf = pd.read_csv(DATA / "esm2_t6_8M_embeddings_shuffled.csv.gz").set_index("accession")
print("proteins:", prot.shape, "| embeddings:", emb.shape, "| shuffled-sequence embeddings:", emb_shuf.shape)
prot.drop(columns=["sequence", "shuffled_sequence"]).head()

**1a.** Describe the data with code: the distribution of sequence `length` (summary and a
histogram, split by each label), the missing values in every column, how many proteins carry
each label and how many carry both (`pd.crosstab`), and whether any sequence appears twice.

In [ ]:
# your code here


**1b.** Where do the labels come from? The provided cell counts the strongest evidence code
behind each annotation (see `data/README.md`).

In [ ]:
# provided
sp = prot["has_signal_peptide"].values
tm = prot["is_transmembrane"].values
print("signal-peptide labels by evidence:\n", prot.loc[sp == 1, "sp_evidence"].value_counts(), "\n")
print("transmembrane labels by evidence:\n", prot.loc[tm == 1, "tm_evidence"].value_counts())

**1c. The split (provided).** One stratified 75/25 split on both labels together, seed 42:
the same split as the Week 7 lecture and website. Every model in this lab uses these indices.

In [ ]:
# provided
idx = np.arange(len(prot))
train_idx, test_idx = train_test_split(idx, test_size=0.25, stratify=2 * tm + sp, random_state=SEED)
CV = StratifiedKFold(5, shuffle=True, random_state=SEED)
pd.DataFrame({"train": [len(train_idx), sp[train_idx].sum(), tm[train_idx].sum()],
              "test": [len(test_idx), sp[test_idx].sum(), tm[test_idx].sum()]},
             index=["proteins", "signal peptide", "transmembrane"])

**Question 1.** (a) What fraction of proteins carry each label, and what does that imply for
accuracy as a metric? (b) Most labels come from "sequence analysis". What does that mean, and
why might it matter when the *inputs* to our models are also computed from the sequence?
(c) A protein with `has_signal_peptide = 0`: has it been shown to lack a signal peptide?

*Your answer here.*

## Task 2. Classical protein descriptors (about 10 minutes)

The simplest fixed-length description of a protein is its **amino-acid composition**: the
fraction of each of the 20 residues. It ignores order completely. We add three more descriptors:
**log length**, and two summaries of the **Kyte–Doolittle hydropathy** scale (Kyte and Doolittle,
1982): the highest mean hydropathy over any 19-residue window (a membrane-spanning helix is about
that long and strongly hydrophobic) and the mean hydropathy of the first 25 residues (where a
signal peptide's hydrophobic core sits). Unlike composition, the last two depend on *where*
residues are.

**2a.** Using the scale `KD` below, write `max_hydropathy(seq, w=19)` and
`nterm_hydropathy(seq, n=25)`, and compute both for the first protein.

In [ ]:
# provided: the Kyte-Doolittle hydropathy scale (J Mol Biol 1982;157:105-132)
KD = {"A": 1.8, "R": -4.5, "N": -3.5, "D": -3.5, "C": 2.5, "Q": -3.5, "E": -3.5, "G": -0.4, "H": -3.2,
      "I": 4.5, "L": 3.8, "K": -3.9, "M": 1.9, "F": 2.8, "P": -1.6, "S": -0.8, "T": -0.7, "W": -0.9,
      "Y": -1.3, "V": 4.2}
AA = "ACDEFGHIKLMNPQRSTVWY"

In [ ]:
# your code here


**2b. The feature matrices (provided).** The module's reference implementation builds the
23 classical descriptors for every protein (and for the shuffled sequences, used in Task 8). Check
that your two functions agree with columns 21 and 22 for the first protein.

In [ ]:
# provided
FEATURES = [f"frac_{a}" for a in AA] + ["log_length", "max_hydropathy_19", "nterm_hydropathy_25"]

def classical_features(seqs, w=19, n=25):
    rows = []
    for q in seqs:
        h = np.array([KD[a] for a in q])
        rows.append([q.count(a) / len(q) for a in AA] +
                    [np.log(len(q)), np.convolve(h, np.ones(w) / w, mode="valid").max(), h[:n].mean()])
    return np.array(rows)

X_classical = classical_features(prot["sequence"])
X_classical_shuf = classical_features(prot["shuffled_sequence"])
X_comp = X_classical[:, :20]                      # composition only
print(X_classical.shape, "| first protein:", np.round(X_classical[0, 20:], 3))
pd.DataFrame(X_classical, columns=FEATURES).describe().T.round(3).iloc[[0, 9, 13, 20, 21, 22]]

**Question 2.** Which of the 23 descriptors are blind to residue order, and which are not?
Name one biological property of a protein that none of them can capture.

*Your answer here.*

## Task 3. PCA on the classical descriptors (about 15 minutes)

PCA finds orthogonal directions of greatest variance. It uses no labels, so here we fit it on
all 2,000 proteins to *describe* the data (when PCA feeds a model, it goes inside the Pipeline:
Extension A).

**Predict first:** without standardising, which descriptors will dominate PC1, and why?

**3a.** Standardise `X_classical` and fit `PCA()`. Report PC1's and PC2's share of the variance
and the number of components needed for 90%. Plot the explained-variance ratio and the
cumulative curve.

**3b.** Plot PC1 against PC2, coloured by signal peptide and, separately, by transmembrane.
List the six largest PC1 loadings (with sign).

**3c.** Repeat PCA *without* standardising. What share does PC1 take now, and which features
dominate it?

In [ ]:
# your code here


**Question 3.** (a) Is the standardised spectrum steep or flat, and what does that say about
redundancy among the descriptors? (b) What biological contrast does PC1 express? (c) Explain the
unscaled result: is it telling you about the proteins or about the units?

*Your answer here.*

## Task 4. t-SNE and UMAP (about 15 minutes)

t-SNE and UMAP place points in 2-D so that near neighbours stay near. They are stochastic and
have settings that change the picture. **Trustworthiness** (between 0 and 1) measures how many
of each point's nearest neighbours in the 2-D map are true neighbours in the original space.

**Predict first:** will a 2-D t-SNE map keep a protein's true neighbours better or worse than
the first two principal components?

**4a. (provided)** The cell computes t-SNE with perplexity 5 and 50, and UMAP with
`n_neighbors` 5 and 50, on the standardised classical descriptors (about 30 seconds).

**4b.** Plot the four maps and the PCA map side by side, coloured by transmembrane, and compute
the trustworthiness (`n_neighbors=10`) of each against `Z_cl`.

In [ ]:
# provided
Z_cl = StandardScaler().fit_transform(X_classical)
maps_cl = {"PCA": PCA(n_components=2, random_state=SEED).fit_transform(Z_cl)}
for perp in [5, 50]:
    maps_cl[f"t-SNE, perplexity {perp}"] = TSNE(perplexity=perp, init="pca", random_state=0).fit_transform(Z_cl)
if HAVE_UMAP:
    for nn in [5, 50]:
        maps_cl[f"UMAP, n_neighbors {nn}"] = umap.UMAP(n_neighbors=nn, min_dist=0.1, random_state=0,
                                                       n_jobs=1).fit_transform(Z_cl)
print(list(maps_cl))

In [ ]:
# your code here


**Question 4.** (a) Which map keeps neighbours best, and how does perplexity (or
`n_neighbors`) change the picture? (b) Three things you **cannot** read from a t-SNE or UMAP map.
(c) If transmembrane proteins form a visible group, does that show the descriptors "understand"
membrane proteins?

*Your answer here.*

## Task 5. The ESM-2 embeddings (about 10 minutes)

ESM-2 is a protein language model pretrained to predict masked residues from the rest of the
sequence (no annotations are used). The module team ran the 8-million-parameter model
(`facebook/esm2_t6_8M_UR50D`, pinned revision; see `data/PROVENANCE.json`) on every sequence
and **mean-pooled** the final layer over residue positions: one 320-number vector per protein,
whatever its length.

**5a. (provided)** Put the embeddings in the same row order as `prot` and check integrity.

In [ ]:
# provided
E = emb.loc[prot["accession"]].values
E_shuf = emb_shuf.loc[prot["accession"]].values
print("shape", E.shape, "| non-finite values", (~np.isfinite(E)).sum(),
      "| accessions match", set(emb.index) == set(prot["accession"]),
      "| identical real and shuffled rows", int((np.abs(E - E_shuf).sum(1) == 0).sum()))

**5b.** (i) Anisotropy: compute the mean cosine similarity between all pairs of raw
embeddings, then after subtracting the mean vector. (ii) Standardise `E`, fit `PCA()`, and
report how many components are needed for 90% of the variance. (iii) Plot PC1 against PC2 and
(if UMAP is available) a UMAP (`n_neighbors=15, min_dist=0.1, random_state=0`) coloured by signal
peptide, with the trustworthiness of each.

In [ ]:
# your code here


**Question 5.** (a) What does a mean raw cosine near 1 mean for cosine-based nearest
neighbours, and what is the fix? (b) The embeddings were computed for all 2,000 proteins
*before* the train/test split. Why is that not leakage, when fitting a `StandardScaler` on all
2,000 would be? (c) Compare the ESM-2 spectrum with the classical one (Task 3).

*Your answer here.*

## Task 6. The key experiment: k-NN on classical descriptors versus ESM-2 (about 30 minutes)

You have studied k-NN, distance metrics and scaling. Now use them to compare two representations
**fairly**. Everything is held fixed except the representation:

| Held fixed | Value |
|---|---|
| Proteins and labels | the same 2,000; the same target |
| Split | `train_idx` / `test_idx` from Task 1 (1,500 / 500) |
| Model selection | 5-fold stratified CV on training proteins only (`CV`), mean ROC AUC |
| Pipeline | optional `StandardScaler` → `KNeighborsClassifier`, refitted in every fold |
| Grid | k in `K_GRID` × metric {euclidean, manhattan, cosine} × weights {uniform, distance} × scaling {none, standardise}: 192 settings |
| Test set | used once per representation, for the selected pipeline only |

**Predict first:** for the signal-peptide task, will 320 learned dimensions beat 23 hand-made
descriptors? For the transmembrane task?

**6a.** Write the pipeline and the `GridSearchCV` for the **classical descriptors** and the
**signal-peptide** label, fitted on the training proteins only. Print `best_params_` and
`best_score_`.

In [ ]:
# provided: the one grid every representation gets
K_GRID = [1, 3, 5, 7, 9, 11, 15, 21, 25, 31, 35, 41, 51, 61, 75, 101]
GRID = {"scale": ["passthrough", StandardScaler()], "knn__n_neighbors": K_GRID,
        "knn__metric": ["euclidean", "manhattan", "cosine"], "knn__weights": ["uniform", "distance"]}

def make_pipe():
    return Pipeline([("scale", StandardScaler()), ("knn", KNeighborsClassifier())])

In [ ]:
# your code here


**6b. (provided)** The same search for every representation and both tasks:
composition only (20), classical (23), ESM-2 (320), and ESM-2 of the shuffled sequences (the
control for Task 8). About a minute.

In [ ]:
# provided
REPS = {"composition": X_comp, "classical": X_classical, "esm2": E, "esm2_shuffled": E_shuf}
TASKS = {"signal_peptide": sp, "transmembrane": tm}

def describe(p):
    return (f"k={p['knn__n_neighbors']}, {p['knn__metric']}, {p['knn__weights']}, "
            f"{'unscaled' if isinstance(p['scale'], str) else 'standardised'}")

searches, rows = {}, []
t0 = time.time()
for task, y in TASKS.items():
    for rep, M in REPS.items():
        s = GridSearchCV(make_pipe(), GRID, cv=CV, scoring="roc_auc", n_jobs=-1).fit(M[train_idx], y[train_idx])
        searches[(task, rep)] = s
        sd = s.cv_results_["std_test_score"][s.best_index_]
        rows.append({"task": task, "representation": rep, "dimensions": M.shape[1],
                     "CV ROC AUC": round(s.best_score_, 4), "CV SD": round(sd, 4), "selected": describe(s.best_params_)})
cv_table = pd.DataFrame(rows).set_index(["task", "representation"])
print("%.0f s" % (time.time() - t0))
cv_table

**6c. The test set, once (provided).** Each representation's selected pipeline (refitted on
all 1,500 training proteins by `GridSearchCV`) predicts the 500 test proteins. A paired bootstrap
gives a 95% interval for the difference in test ROC AUC.

In [ ]:
# provided
def bootstrap_diff(y, pa, pb, n=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    d = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() == y[i].max():
            continue
        d.append(roc_auc_score(y[i], pa[i]) - roc_auc_score(y[i], pb[i]))
    return roc_auc_score(y, pa) - roc_auc_score(y, pb), np.percentile(d, [2.5, 97.5])

test_rows, probs, preds = [], {}, {}
for (task, rep), s in searches.items():
    y = TASKS[task]
    probs[(task, rep)] = s.predict_proba(REPS[rep][test_idx])[:, 1]
    preds[(task, rep)] = s.predict(REPS[rep][test_idx])
    test_rows.append({"task": task, "representation": rep,
                      "ROC AUC": roc_auc_score(y[test_idx], probs[(task, rep)]),
                      "avg precision": average_precision_score(y[test_idx], probs[(task, rep)]),
                      "precision": precision_score(y[test_idx], preds[(task, rep)], zero_division=0),
                      "recall": recall_score(y[test_idx], preds[(task, rep)]),
                      "F1": f1_score(y[test_idx], preds[(task, rep)])})
test_table = pd.DataFrame(test_rows).set_index(["task", "representation"]).round(3)
display(test_table)
for task, y in TASKS.items():
    for other in ["classical", "composition"]:
        d, ci = bootstrap_diff(y[test_idx], probs[(task, "esm2")], probs[(task, other)])
        print(f"{task:15s} ESM-2 minus {other:12s}: {d:+.3f}  95% interval [{ci[0]:+.3f}, {ci[1]:+.3f}]")

**6d.** Plot the four confusion matrices (classical and ESM-2, both tasks) for the test
set. For the signal-peptide task, explain why recall at scikit-learn's default threshold differs
so much between representations even where the ROC AUC is similar.

In [ ]:
# your code here


**Question 6.** (a) Report, for each task, the selected settings and the test ROC AUC of
the classical and ESM-2 models, with the bootstrap interval. Which representation did better, for
which task? (b) Is the comparison fair? Name every way in which it is, and one way in which the
two representations still differ besides the information they contain. (c) Why did GridSearchCV
choose different metrics and scalings for different representations, and is that a problem?

*Your answer here.*

**Question 6d.** Explain the recall difference.

*Your answer here.*

## Task 7. Whose neighbours? Comparing neighbourhoods (about 10 minutes)

k-NN predicts from neighbours, so compare the neighbours themselves. For every protein, find its
10 nearest proteins (among all 2,000; standardised features, Euclidean) in several spaces.

**7a. (provided)** The helper returns each protein's 10 nearest neighbours.

In [ ]:
# provided
def neighbours(M, k=10, metric="euclidean"):
    nn = NearestNeighbors(n_neighbors=k + 1, metric=metric).fit(M)
    _, idx_ = nn.kneighbors(M)
    return np.array([[j for j in row if j != i][:k] for i, row in enumerate(idx_)])

def overlap(A, B):
    return np.mean([len(set(a) & set(b)) / len(a) for a, b in zip(A, B)])

Z_e = StandardScaler().fit_transform(E)
pcs_e = PCA(random_state=SEED).fit_transform(Z_e)       # unsupervised: all proteins
nbr = {"composition": neighbours(StandardScaler().fit_transform(X_comp)),
       "classical": neighbours(StandardScaler().fit_transform(X_classical)),
       "esm2": neighbours(Z_e)}
for kc in [2, 10, 50]:
    nbr[f"esm2_pca{kc}"] = neighbours(pcs_e[:, :kc])
print({k: v.shape for k, v in nbr.items()})

**7b.** (i) Make a table: for each space, the mean overlap of its neighbour sets with the
ESM-2 neighbour sets, and the mean share of a **signal-peptide** protein's neighbours that also
have a signal peptide. (ii) Print the 10 neighbours of Interleukin-12 subunit beta (P29460, a
secreted protein with an experimentally supported signal peptide) in the composition, classical
and ESM-2 spaces, with their names and labels.

In [ ]:
# your code here


**Question 7.** (a) How similar are the neighbourhoods defined by the classical descriptors
and by ESM-2? (b) What does PCA do to the ESM-2 neighbourhoods as the number of components
falls? (c) Interpret the IL-12B example biologically, carefully.

*Your answer here.*

## Task 8. The residue-shuffling control (about 10 minutes)

Every protein was also embedded after **shuffling its residues**: the same residues, so the same
composition and length, in random order. If ESM-2's advantage comes from order, it should
vanish for the shuffled sequences.

**Predict first:** after shuffling, will k-NN on the ESM-2 embedding fall to the level of
composition, stay at its original level, or fall below composition?

**8a.** (i) Check that composition and length are unchanged by the shuffle and measure how the
two hydropathy features change (`X_classical` against `X_classical_shuf`). (ii) In the
standardised ESM-2 space (scaler fitted on the real embeddings), compare each protein's distance
to its own shuffled copy with its distance to its nearest *other* real protein. (iii) Use
`cv_table` and `test_table` to compare ESM-2 and shuffled ESM-2.

In [ ]:
# your code here


**Question 8.** What does the control show, and what does it **not** show?

*Your answer here.*

## Task 9. Critical reflection (about 10 minutes)

Answer in short paragraphs, using your numbers.

1. Which representation performed better in this experiment, for which task, and how sure are you?
2. What information might explain the difference (and the absence of a difference)?
3. How did dimensionality reduction affect nearest-neighbour relationships?
4. Did visual clusters in the maps correspond to the biological annotations?
5. What are the limitations of the dataset and of the evaluation? Consider homology between
   training and test proteins, pretraining exposure, and annotation uncertainty.
6. What further experiment would you run next, and what would it tell you?

*Your answer here.*

---
## Checkpoint

You have finished the core lab when Tasks 1–9 run top to bottom and every question is answered.
The extensions are optional.

## Extension A. Dimensionality reduction inside the k-NN pipeline

Put `StandardScaler → PCA(n_components=k) → KNeighborsClassifier` (using ESM-2's selected
signal-peptide setting) in one Pipeline and plot cross-validated ROC AUC for k in 2, 5, 10, 20,
50, 100. If UMAP is available, try `StandardScaler → UMAP(2) → KNeighborsClassifier` (Euclidean)
too. Why can t-SNE not be used this way?

In [ ]:
# your code here


*Your answer here.*

## Extension B. Choosing a threshold without touching the test set

For the signal-peptide ESM-2 and classical models, get cross-validated probabilities on the
**training** proteins (`cross_val_predict(..., method="predict_proba")`), choose the threshold
that maximises F1 there, and only then apply it to the test probabilities from Task 6c.

In [ ]:
# your code here


*Your answer here.*

## Extension C. Annotation evidence and related sequences

(i) For the signal-peptide test proteins, compare the recall of the classical and ESM-2 models
separately for experimentally supported and for "sequence analysis" labels. (ii) Measure, for
each test protein, the largest share of its distinct 5-mers found in any single training protein.
How many test proteins have a close relative in the training set?

In [ ]:
# your code here


*Your answer here.*

## Extension D. Regenerating the embeddings (instructions only; needs a network and PyTorch)

The embeddings in `data/` were produced by the module team with the procedure below, recorded
in `data/PROVENANCE.json` (model, revision, pooling, library versions, checksums). Running it is
**not** part of the lab: it needs `torch` and `transformers`, downloads the model (about 30 MB),
and takes about 1–2 minutes on a laptop CPU for 2,000 proteins (a GPU is not needed for the 8M
model; larger ESM-2 models need much more memory).

```python
import torch
from transformers import AutoModel, AutoTokenizer
MODEL, REVISION = "facebook/esm2_t6_8M_UR50D", "c731040fcd8d73dceaa04b0a8e6329b345b0f5df"
tok = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
model = AutoModel.from_pretrained(MODEL, revision=REVISION).eval()
with torch.no_grad():
    enc = tok([seq], return_tensors="pt")
    h = model(**enc).last_hidden_state[0]          # (tokens, 320): <cls>, residues, <eos>
    vector = h[1:len(seq) + 1].mean(0).numpy()     # mean over residue positions only
```

Compare one protein's vector with the frozen one (`emb.loc["P29460"]`): differences should be
below about 1e-4 (the frozen values are rounded to 5 decimals).